In [3]:
import os
import pyabf
import numpy as np
import matplotlib.pyplot as plt

In [4]:
#set dir

directory = "/Users/jayashri/Desktop/06122025"
abf = pyabf.ABF("/Users/jayashri/Desktop/06122025/25612002Events.abf")

In [5]:
#First, given a directory, find all abf filenames with the string "events" in the filename.

In [6]:
#Second, for each file plot the average waveform across all sweeps.

In [7]:
#Third, for each file, remove any sweep which is obviously a membrane test and not a real synaptic event. 

In [8]:
#Fourth, for each file, plot the average waveform, now with the membrane tests removed. 

In [9]:
#Fifth, save the plot generated for each file in step 4, with the filename as the name of the image and the title of the plot. 

In [10]:
def process_abf_file(filepath, output_dir="plots"):
    abf = pyabf.ABF(filepath)
    filename = os.path.basename(filepath).replace(".abf", "")

    # First plot: average of all sweeps
    avg_all = get_average_waveform(abf)

    # Second plot: average with membrane tests removed
    avg_filtered, kept_count = remove_membrane_tests(abf)

    # Plot both waveforms
    plt.figure(figsize=(10, 6))
    plt.plot(avg_all, label="All Sweeps", linestyle="--", alpha=0.5)
    plt.plot(avg_filtered, label=f"Filtered Sweeps ({kept_count}/{abf.sweepCount})", linewidth=2)
    plt.title(filename)
    plt.xlabel("Time (points)")
    plt.ylabel("Amplitude (pA or mV)")
    plt.legend()
    plt.tight_layout()

    # Save figure
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, f"{filename}.png"))
    plt.close()

def main(directory):
    files = find_abf_files_with_events(directory)
    for filepath in files:
        process_abf_file(filepath)

In [11]:
def find_abf_files_with_events(directory):
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".abf") and "events" in f.lower()]

In [12]:
find_abf_files_with_events(directory)

['/Users/jayashri/Desktop/06122025/25612008Events.abf',
 '/Users/jayashri/Desktop/06122025/25612002Events.abf']

In [13]:
def get_average_waveform(abf):
    sweep_data = []
    for i in range(abf.sweepCount):
        abf.setSweep(i)
        sweep_data.append(abf.sweepY)
    avg_waveform = np.mean(sweep_data, axis=0)
    return avg_waveform, len(sweep_data)  # This is the number of sweeps

In [14]:
get_average_waveform(abf)

(array([ 0.    ,  0.2799, -0.0165, ...,  0.8919,  1.0024,  0.9077],
       dtype=float32),
 1882)

In [27]:
def remove_membrane_tests(abf, slope_threshold=2000):
    """
    Removes sweeps where the maximum absolute slope (first derivative) exceeds a threshold.
    These are likely membrane tests with sharp transitions.
    
    Parameters:
    - abf: pyabf.ABF object
    - slope_threshold: max(abs(dY/dt)) allowed for inclusion

    Returns:
    - avg_waveform: average of included sweeps
    - num_included: number of sweeps used in the average
    - excluded_indices: list of indices excluded
    """
    filtered_sweeps = []
    excluded_indices = []

    for i in range(abf.sweepCount):
        abf.setSweep(i)
        dy = np.diff(abf.sweepY) / np.diff(abf.sweepX)
        max_slope = np.max(np.abs(dy))

        if max_slope < slope_threshold:
            filtered_sweeps.append(abf.sweepY)
        else:
            excluded_indices.append(i)

    if not filtered_sweeps:
        raise ValueError("No sweeps passed the slope threshold. Try increasing the slope_threshold.")

    avg_waveform = np.mean(filtered_sweeps, axis=0)
    return avg_waveform, len(filtered_sweeps), excluded_indices

In [29]:
remove_membrane_tests(abf, slope_threshold = 2000)

ValueError: No sweeps passed the slope threshold. Try increasing the slope_threshold.

In [ ]:
# Example usage
if __name__ == "__main__":
    main("/Users/jayashri/Desktop/06122025")